# Challenge 4 — Limpieza de Datos

**Nombre:** Miriam Ivonne Castañeda Chavarría
**Clase:** Limpieza de Datos  
**Dataset:** `ventas.csv`

## Objetivo
Revisar el dataset, detectar problemas y hacer una limpieza conservadora, sin modificar información de manera arbitraria.

## 1. Cargar los datos

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/ivonnecastaneda/Downloads/ventas.csv")
df.head()

,fecha,region_id,vendedor,producto_id,cantidad,nombre_producto,categoria,precio_unitario,total
0,2023-01-01,1,Vendedor_4,7,9,Producto 7,Electrónica,1932.73,17394.57
1,2023-01-01,3,Vendedor_10,5,5,Producto 5,Ropa,1240.04,6200.20
2,2023-01-01,5,Vendedor_6,1,1,Producto 1,Electrónica,1264.57,1264.57
3,2023-01-01,4,Vendedor_9,14,3,Producto 14,Deportes,2949.45,8848.35
4,2023-01-01,2,Vendedor_6,17,4,Producto 17,Ropa,1593.14,6372.56


## 2. Diagnóstico inicial
Antes de cambiar datos revisamos información general, faltantes, duplicados y estadísticos.

In [8]:
df.info()

print("Filas y columnas:", df.shape)
print("\nValores faltantes:")
print(df.isna().sum())
print("\nDuplicados:", df.duplicated().sum())

df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   fecha            6000 non-null   object 
 1   region_id        6000 non-null   int64  
 2   vendedor         6000 non-null   object 
 3   producto_id      6000 non-null   int64  
 4   cantidad         6000 non-null   int64  
 5   nombre_producto  6000 non-null   object 
 6   categoria        6000 non-null   object 
 7   precio_unitario  6000 non-null   float64
 8   total            6000 non-null   float64
dtypes: float64(2), int64(3), object(4)
memory usage: 422.0+ KB
Filas y columnas: (6000, 9)

Valores faltantes:
fecha              0
region_id          0
vendedor           0
producto_id        0
cantidad           0
nombre_producto    0
categoria          0
precio_unitario    0
total              0
dtype: int64

Duplicados: 0


,fecha,region_id,vendedor,producto_id,cantidad,nombre_producto,categoria,precio_unitario,total
count,6000,6000.000000,6000,6000.000000,6000.000000,6000,6000,6000.000000,6000.000000
unique,1089,NaN,15,NaN,NaN,20,5,NaN,NaN
top,2024-08-01,NaN,Vendedor_1,NaN,NaN,Producto 4,Deportes,NaN,NaN
freq,14,NaN,459,NaN,NaN,327,2118,NaN,NaN
mean,NaN,2.994167,NaN,10.554333,6.068500,NaN,NaN,1698.233145,10330.677430
std,NaN,1.405867,NaN,5.733486,3.175398,NaN,NaN,885.641029,8153.475421
min,NaN,1.000000,NaN,1.000000,1.000000,NaN,NaN,161.430000,161.530000
25%,NaN,2.000000,NaN,5.000000,3.000000,NaN,NaN,1004.095000,3285.052500
50%,NaN,3.000000,NaN,11.000000,6.000000,NaN,NaN,1826.945000,8214.840000
75%,NaN,4.000000,NaN,16.000000,9.000000,NaN,NaN,2457.115000,15586.277500


### Resultado
El archivo tiene 6,000 filas y 9 columnas. No hay valores faltantes ni filas duplicadas.

## 3. Tabla de revisión

In [9]:
revision = pd.DataFrame({
    "Problema": ["Valores faltantes","Duplicados","Fecha como texto","Texto","Valores imposibles","Total inconsistente","Outliers"],
    "Columna": ["Todas","Todas","fecha","vendedor, nombre_producto, categoria","cantidad, precio_unitario, total","total","total"],
    "Acción": ["Revisar; no hay faltantes","Revisar; no hay duplicados","Convertir a datetime","Quitar espacios","Revisar reglas","Comparar con cantidad × precio","Revisar, no eliminar automáticamente"]})
revision

,Problema,Columna,Acción
0,Valores faltantes,Todas,Revisar; no hay faltantes
1,Duplicados,Todas,Revisar; no hay duplicados
2,Fecha como texto,fecha,Convertir a datetime
3,Texto,"vendedor, nombre_producto, categoria",Quitar espacios
4,Valores imposibles,"cantidad, precio_unitario, total",Revisar reglas
5,Total inconsistente,total,Comparar con cantidad × precio
6,Outliers,total,"Revisar, no eliminar automáticamente"


## 4. Valores faltantes
No hay faltantes, por lo que no fue necesario eliminar filas, columnas ni imputar valores.

In [10]:
df.isna().sum()

fecha              0
region_id          0
vendedor           0
producto_id        0
cantidad           0
nombre_producto    0
categoria          0
precio_unitario    0
total              0
dtype: int64

## 5. Duplicados
No se encontraron filas completamente duplicadas.

In [11]:
print("Duplicados:", df.duplicated().sum())

Duplicados: 0


## 6. Tipos y texto
La columna `fecha` estaba como texto. Se convierte a fecha. También se quitan espacios al inicio y final de las columnas de texto.

In [12]:
df_clean = df.copy()
df_clean["fecha"] = pd.to_datetime(df_clean["fecha"], errors="coerce")

for col in ["vendedor", "nombre_producto", "categoria"]:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print(df_clean.dtypes)
print("Fechas no convertidas:", df_clean["fecha"].isna().sum())

fecha              datetime64[ns]
region_id                   int64
vendedor                   object
producto_id                 int64
cantidad                    int64
nombre_producto            object
categoria                  object
precio_unitario           float64
total                     float64
dtype: object
Fechas no convertidas: 0


## 7. Reglas de negocio

In [13]:
print("Cantidad <= 0:", (df_clean["cantidad"] <= 0).sum())
print("Precio unitario <= 0:", (df_clean["precio_unitario"] <= 0).sum())
print("Total < 0:", (df_clean["total"] < 0).sum())

diferencia = (df_clean["total"] - df_clean["cantidad"] * df_clean["precio_unitario"]).abs()
print("Total con diferencia > 0.01:", (diferencia > 0.01).sum())
print("Mayor diferencia:", diferencia.max())

Cantidad <= 0: 0
Precio unitario <= 0: 0
Total < 0: 0
Total con diferencia > 0.01: 0
Mayor diferencia: 7.275957614183426e-12


### Resultado
No hay cantidades, precios o totales negativos. Además, `total` coincide con `cantidad × precio_unitario` dentro de una tolerancia de 0.01.

## 8. Outliers
Se revisan con IQR. Un outlier no significa automáticamente que sea un error, así que no se elimina sin evidencia.

In [14]:
q1 = df_clean["total"].quantile(0.25)
q3 = df_clean["total"].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers = df_clean[(df_clean["total"] < limite_inferior) | (df_clean["total"] > limite_superior)]
print("Límite inferior:", round(limite_inferior, 2))
print("Límite superior:", round(limite_superior, 2))
print("Posibles outliers:", len(outliers))
outliers.head()

Límite inferior: -15166.79
Límite superior: 34038.12
Posibles outliers: 16


,fecha,region_id,vendedor,producto_id,cantidad,nombre_producto,categoria,precio_unitario,total
325,2023-03-05,1,Vendedor_12,13,11,Producto 13,Deportes,3249.71,35746.81
499,2023-04-04,2,Vendedor_12,13,11,Producto 13,Deportes,3270.15,35971.65
851,2023-06-05,4,Vendedor_13,13,11,Producto 13,Deportes,3162.60,34788.60
1371,2023-09-03,3,Vendedor_3,13,11,Producto 13,Deportes,3298.61,36284.71
1429,2023-09-15,3,Vendedor_4,13,11,Producto 13,Deportes,3290.59,36196.49


**Decisión:** conservar los outliers. Pueden ser ventas reales de monto alto y no hay evidencia suficiente para borrarlos.

## 9. Validación final

In [15]:
print("Filas finales:", len(df_clean))
print("Columnas finales:", len(df_clean.columns))
print("Duplicados:", df_clean.duplicated().sum())
print("\nFaltantes:")
print(df_clean.isna().sum())
print("\nTipos:")
print(df_clean.dtypes)

Filas finales: 6000
Columnas finales: 9
Duplicados: 0

Faltantes:
fecha              0
region_id          0
vendedor           0
producto_id        0
cantidad           0
nombre_producto    0
categoria          0
precio_unitario    0
total              0
dtype: int64

Tipos:
fecha              datetime64[ns]
region_id                   int64
vendedor                   object
producto_id                 int64
cantidad                    int64
nombre_producto            object
categoria                  object
precio_unitario           float64
total                     float64
dtype: object


## 10. Guardar dataset limpio

In [17]:
df_clean.to_csv("dataset_limpio.csv", index=False, date_format="%Y-%m-%d")
print("Archivo guardado como dataset_limpio.csv")

Archivo guardado como dataset_limpio.csv


## Reflexión final

1. **¿Cuántas filas había inicialmente?** 6,000.
2. **¿Cuántas quedaron?** 6,000, porque no hubo duplicados ni registros que justificaran eliminarse.
3. **¿Qué problema fue más frecuente?** No hubo un problema fuerte de faltantes o duplicados; el ajuste principal fue convertir la fecha.
4. **¿Qué decisión fue la más difícil?** La decisión sobre los outliers, porque un valor alto puede ser una venta real.
5. **¿Qué información pudo perderse?** Si se hubieran eliminado outliers automáticamente, podrían haberse perdido ventas reales. Por eso se conservaron.